In [15]:
import pandas as pd
import duckdb
from IPython.display import HTML
import plotly.graph_objects as go

In [3]:
conn = duckdb.connect("../wynne_strava.db")

In [6]:
df = conn.sql("SELECT * FROM wynne_activitieS ORDER BY start_date_local").df()

In [10]:
df['miles'] = (df['distance'] / 1609.344).round(2)

df['start_date_local'] = pd.to_datetime(df['start_date_local'])
df['pace_minutes'] = (df['elapsed_time'] / 60) / df['miles']
df = df.sort_values('start_date_local')

In [11]:
df.head(2)

,id,name,sport_type,distance,elapsed_time,total_elevation_gain,start_date_local,achievement_count,kudos_count,average_speed,max_speed,pr_count,miles,pace_minutes
0,8550343253,Morning Run,Run,3810.1,1305,0.0,2023-02-12 09:51:06+00:00,3,0,2.920,4.162,1,2.37,9.177215
1,8578164631,Afternoon Run,Run,4992.4,1874,25.6,2023-02-17 15:02:04+00:00,0,0,2.761,4.168,0,3.10,10.075269


In [12]:
plot_df = df[(df['pace_minutes'] >= 5) & (df['pace_minutes'] <= 15)]

In [16]:
def format_pace(minutes):
    m = int(minutes)
    s = int((minutes % 1) * 60)
    return f"{m}:{s:02d}"

fig = go.Figure()
fig.add_trace(go.Scatter(
    x=plot_df['start_date_local'],
    y=plot_df['pace_minutes'],
    mode='lines+markers',
    marker=dict(size=6),
    text=[f"{format_pace(p)} /mi — {d:.2f} mi" for p, d in zip(plot_df['pace_minutes'], plot_df['miles'])],
    hovertemplate='%{x|%b %d, %Y}<br>%{text}<extra></extra>',
))

fig.update_layout(
    title='Mile Pace Over Time',
    xaxis_title='Date',
    yaxis_title='Mile Pace',
    yaxis=dict(
        autorange='reversed',
        tickvals=list(range(6, 16)),
        ticktext=[format_pace(v) for v in range(6, 16)],
    ),
    template='plotly_white',
    height=500,
)